In [ ]:
'''Gemma 4 Developer Agent | Evidence-First Baseline

First runnable package for the Kaggle agent competition. It builds a single-agent
ADK submission ZIP with a short evidence-first workflow and selective code-graph use.

Competition rules: https://www.kaggle.com/competitions/gemma-4-developer-agent/overview
Public implementation used to cross-check ADK YAML shape and workflow ideas:
https://github.com/richie1988/gemma4-evidence-first-swe-agent

This is our own minimal baseline, not an imported submission. It has not been run
inside the competition harness; local checks below validate only package structure.
'''

In [ ]:
'''Set the package destination and the one supported competition model.'''

from pathlib import Path
import zipfile

MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
PACKAGE_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = PACKAGE_DIR / 'submission.zip'
print('Package destination:', SUBMISSION_PATH)


In [ ]:
'''Define the ADK root agent using the public competition tool names.'''

AGENT_YAML = f'''agent_class: LlmAgent
name: evidence_first_baseline
model: {MODEL}
description: >
  Evidence-first software repair agent with selective semantic and graph navigation.
instruction: !include prompts/system.md
tools:
  - name: get_status
  - name: read_file
  - name: edit_file
  - name: write_file
  - name: run_command
  - name: search_similar_code
  - name: get_code_neighbors
  - name: get_code_subgraph
  - name: submit_patch
'''

SYSTEM_PROMPT = '''You are an autonomous software engineer fixing one repository issue. Produce the smallest correct patch that satisfies the issue and repository tests.

Work from repository evidence. Do not guess file paths or APIs when you can inspect them. Keep changes focused; do not add unrelated refactors, dependencies, generated files, or test weakening.

For each task:
1. Read the issue and turn it into concrete expected behavior and edge cases. Check the remaining time with get_status.
2. Inspect repository status, top-level files, and project test/build instructions with run_command.
3. Use search_similar_code when the issue describes behavior without naming the implementation. Verify candidates with read_file or exact text search.
4. Once a likely symbol is known, use get_code_neighbors to inspect relevant callers or dependencies. Use get_code_subgraph only when a small set of symbols has a meaningful relationship to verify. Treat graph results as navigation hints, not proof.
5. Reproduce the failure with the narrowest relevant test or command when practical. State a concrete root-cause hypothesis before editing.
6. Make the smallest behaviorally complete change. Inspect surrounding code and preserve local conventions.
7. Run the focused test first, then a reasonable nearby regression check if time allows. Do not spend the task budget on broad unrelated test suites.
8. Review git status and git diff. Remove scratch files and accidental edits. Run git diff --check when available.
9. Call submit_patch only when the diff is coherent and addresses the issue. If you cannot establish a safe fix, do not submit speculative changes; report what evidence is missing.

Use the competition tools only. Keep the work bounded by the task time limit and preserve time for validation.
'''

In [ ]:
'''Build submission.zip with agent.yaml at the archive root.'''

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr('agent.yaml', AGENT_YAML)
    archive.writestr('prompts/system.md', SYSTEM_PROMPT)

print('Wrote', SUBMISSION_PATH, 'bytes:', SUBMISSION_PATH.stat().st_size)


In [ ]:
'''Check archive paths and required competition configuration without external packages.'''

with zipfile.ZipFile(SUBMISSION_PATH) as archive:
    names = set(archive.namelist())
    assert 'agent.yaml' in names, 'agent.yaml must be at the ZIP root'
    assert 'prompts/system.md' in names, 'system prompt is missing'
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in names), 'Unsafe path in archive'
    agent_text = archive.read('agent.yaml').decode('utf-8')
    prompt_text = archive.read('prompts/system.md').decode('utf-8')

assert MODEL in agent_text
assert 'instruction: !include prompts/system.md' in agent_text
assert 'name: submit_patch' in agent_text
assert 'submit_patch' in prompt_text
print('Local package checks passed. This does not replace a Kaggle harness run.')
print('Archive files:', sorted(names))
